# Vision Transformers (ViT): Teoría y Práctica

## Fundamentos Teóricos

### ¿Qué es un Vision Transformer?

El Vision Transformer (ViT) es una arquitectura que aplica el mecanismo de **self-attention** de los Transformers (originalmente diseñados para NLP) directamente a imágenes. Fue introducido en el paper "An Image is Worth 16x16 Words" (Dosovitskiy et al., 2021).

### Limitaciones de las CNN
- Las CNN tienen un **campo receptivo local** (capturan patrones cercanos)
- Para capturar relaciones globales, necesitan muchas capas convolucionales
- El campo receptivo crece lentamente con la profundidad

### Ventajas de los Transformers en Visión
- **Self-attention**: Cada parche puede atender a cualquier otro parche (relaciones globales)
- **Escalabilidad**: Funciona mejor con grandes cantidades de datos
- **Arquitectura uniforme**: Misma arquitectura que NLP, facilitando multimodalidad

### Arquitectura del ViT

```
Imagen (224x224) → Parches (16x16, 196 parches)
         ↓
Aplanar y proyectar a embeddings (196 × D)
         ↓
Añadir positional encoding
         ↓
Transformer Encoder (L capas)
         ↓
Token [CLS] → MLP Head → Clasificación
```

### Pasos Detallados:

1. **Patchify**: La imagen se divide en parches de tamaño fijo (ej: 16×16).
   - Para una imagen de 224×224 y parches 16×16: 196 parches.

2. **Linear Projection**: Cada parche se aplana y se proyecta a un vector de dimensión D usando una capa lineal.

3. **Positional Encoding**: Se añade información de posición porque la atención es invariante a la posición.

4. **Token [CLS]**: Se añade un token especial al inicio (como en BERT) cuya salida final se usa para clasificación.

5. **Transformer Encoder**: Capas de Multi-Head Self-Attention + MLP + LayerNorm + Residual connections.

6. **MLP Head**: Clasificador final (LayerNorm + Dense + Tanh + Dense + Softmax).

### Fórmula de Self-Attention:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Donde:
- Q (Query), K (Key), V (Value) son proyecciones de los embeddings
- $d_k$ es la dimensión de las keys (factor de escala)

## Implementación Práctica con PyTorch y Hugging Face

Esta implementación usa el modelo pre-entrenado `google/vit-base-patch16-224-in21k` y lo fine-tunea en CIFAR-10.

In [1]:
#!pip install transformers datasets torchvision torch scikit-learn matplotlib seaborn tqdm


In [1]:
# ==================== INSTALACIÓN ====================
# %%capture
# !pip install transformers datasets torchvision torch scikit-learn matplotlib seaborn tqdm

# ==================== IMPORTACIÓN DE LIBRERÍAS ====================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomRotation
from datasets import load_dataset
from transformers import ViTForImageClassification, ViTImageProcessor, TrainingArguments, Trainer

# Verificar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ==================== CARGA DEL MODELO Y FEATURE EXTRACTOR ====================

# Nombre del modelo pre-entrenado
model_name = "google/vit-base-patch16-224-in21k"
num_classes = 10  # CIFAR-10

# Cargar feature extractor (normalización, tamaño esperado, etc.)
feature_extractor = ViTImageProcessor.from_pretrained(model_name)
print(f"Feature extractor cargado - Tamaño de imagen: {feature_extractor.size}")
print(f"Normalización - mean: {feature_extractor.image_mean}, std: {feature_extractor.image_std}")

# Cargar modelo pre-entrenado
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    ignore_mismatched_sizes=True  # Permite cambiar la cabeza de clasificación
)
model.to(device)

# Mostrar información del modelo
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parámetros: {total_params / 1e6:.2f}M")
print(f"Parámetros entrenables: {trainable_params / 1e6:.2f}M")
print(f"Parámetros congelados: {(total_params - trainable_params) / 1e6:.2f}M")

Feature extractor cargado - Tamaño de imagen: SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)
Normalización - mean: (0.5, 0.5, 0.5), std: (0.5, 0.5, 0.5)


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 5500.51it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Total de parámetros: 85.81M
Parámetros entrenables: 85.81M
Parámetros congelados: 0.00M


In [ ]:
# ==================== PREPARACIÓN DEL DATASET CIFAR-10 ====================

# Cargar dataset
dataset = load_dataset("cifar10")
print(dataset)

# Clases de CIFAR-10
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
print(f"Clases: {cifar10_classes}")

# Transformaciones para entrenamiento (con aumento de datos)
def train_transforms(example_batch):
    transforms = Compose([
        Resize((224, 224)),
        RandomHorizontalFlip(p=0.5),
        RandomRotation(degrees=10),
        ToTensor(),
        Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
    ])
    example_batch['pixel_values'] = [transforms(img.convert("RGB")) for img in example_batch['img']]
    return example_batch

# Transformaciones para evaluación (sin aumento)
def eval_transforms(example_batch):
    transforms = Compose([
        Resize((224, 224)),
        ToTensor(),
        Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
    ])
    example_batch['pixel_values'] = [transforms(img.convert("RGB")) for img in example_batch['img']]
    return example_batch

# Aplicar transformaciones
prepared_dataset = dataset
prepared_dataset["train"] = prepared_dataset["train"].map(train_transforms, batched=True)
prepared_dataset["test"] = prepared_dataset["test"].map(eval_transforms, batched=True)

print(f"\nEntrenamiento: {len(prepared_dataset['train'])} imágenes")
print(f"Prueba: {len(prepared_dataset['test'])} imágenes")

DatasetDict({
    train: Dataset({
        features: ['img', 'label'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['img', 'label'],
        num_rows: 10000
    })
})
Clases: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


Map:  20%|██        | 10000/50000 [00:36<02:23, 278.58 examples/s]

In [ ]:
# ==================== VISUALIZACIÓN DE DATOS ====================

def show_images(images, labels, predictions=None, num_images=6):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
    for i in range(num_images):
        img = images[i].permute(1, 2, 0).numpy()
        # Desnormalizar
        mean = np.array(feature_extractor.image_mean)
        std = np.array(feature_extractor.image_std)
        img = std * img + mean
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        title = cifar10_classes[labels[i]]
        if predictions is not None:
            title += f"\nPred: {cifar10_classes[predictions[i]]}"
        axes[i].set_title(title)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Mostrar ejemplos del dataset
sample = dataset['test'].select(range(8))
sample_images = []
sample_labels = []
for item in sample:
    img = item['img']
    sample_images.append(img)
    sample_labels.append(item['label'])

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(sample_images[i])
    plt.title(cifar10_classes[sample_labels[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CONFIGURACIÓN DE ENTRENAMIENTO ====================

training_args = TrainingArguments(
    output_dir='./vit_cifar10_results',
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True if torch.cuda.is_available() else False,
    report_to="none",  # No reportar a wandb/tensorboard
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
)

# Función de métrica
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Crear Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

print("Configuración de entrenamiento lista")
print(f"Batch size entrenamiento: {training_args.per_device_train_batch_size}")
print(f"Batch size evaluación: {training_args.per_device_eval_batch_size}")
print(f"Épocas: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")

In [ ]:
# ==================== ENTRENAMIENTO (OPCIONAL - DESCOMENTAR PARA EJECUTAR) ====================

# Nota: El entrenamiento completo puede tomar varios minutos.
# Para ejecutar, descomentar la siguiente línea:

# trainer.train()

# Si no se ejecuta el entrenamiento, cargamos un modelo pre-entrenado de ejemplo
print("⚠️ Entrenamiento omitido para rapidez. En un entorno real, descomentar trainer.train()")
print("Los resultados mostrados a continuación son simulados basados en rendimiento típico.")

# Simular métricas del entrenamiento (para demostración)
mock_history = {
    'loss': [1.8, 1.2, 0.9, 0.7, 0.5],
    'accuracy': [0.45, 0.65, 0.75, 0.82, 0.86],
    'val_loss': [1.5, 1.1, 0.85, 0.72, 0.65],
    'val_accuracy': [0.52, 0.68, 0.76, 0.80, 0.83]
}

In [ ]:
# ==================== VISUALIZACIÓN DE CURVAS DE APRENDIZAJE ====================

# Graficar pérdida y exactitud
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de pérdida
axes[0].plot(mock_history['loss'], label='Training Loss', marker='o')
axes[0].plot(mock_history['val_loss'], label='Validation Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curva de Pérdida')
axes[0].legend()
axes[0].grid(True)

# Gráfico de exactitud
axes[1].plot(mock_history['accuracy'], label='Training Accuracy', marker='o')
axes[1].plot(mock_history['val_accuracy'], label='Validation Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Curva de Exactitud')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Mejor exactitud de validación: {max(mock_history['val_accuracy'])*100:.2f}%")

In [ ]:
# ==================== INFERENCIA Y EVALUACIÓN ====================

def predict(model, image):
    """Predice la clase de una imagen preprocesada"""
    model.eval()
    with torch.no_grad():
        if not isinstance(image, torch.Tensor):
            image = torch.tensor(image)
        if image.dim() == 3:
            image = image.unsqueeze(0)
        image = image.to(device)
        outputs = model(pixel_values=image)
        logits = outputs.logits
        pred = logits.argmax(-1).item()
    return pred

# Simular predicciones para evaluación (reemplazar con modelo real)
np.random.seed(42)
y_true_sim = np.random.randint(0, 10, 1000)
# Simular exactitud del 83%
mask = np.random.random(1000) < 0.83
y_pred_sim = np.where(mask, y_true_sim, np.random.randint(0, 10, 1000))

# Métricas de clasificación
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_true_sim, y_pred_sim, target_names=cifar10_classes))

# Matriz de confusión
cm = confusion_matrix(y_true_sim, y_pred_sim)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cifar10_classes, yticklabels=cifar10_classes)
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión - Vision Transformer en CIFAR-10')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== VISUALIZACIÓN DE ATENCIÓN (CONCEPTUAL) ====================

def visualize_attention_heads(model, image, num_heads=12, layer_idx=-1):
    """
    Visualiza los mapas de atención del Transformer.
    Nota: Implementación conceptual - requiere hooks en el modelo real.
    """
    print("\n=== VISUALIZACIÓN DE ATENCIÓN ===")
    print("Los mapas de atención muestran qué regiones de la imagen son más relevantes para la clasificación.")
    print("Cada head de atención puede enfocarse en diferentes patrones: bordes, texturas, formas, objetos completos.")
    print("\nInterpretación:")
    print("- Áreas más brillantes = mayor atención (más relevantes para la decisión)")
    print("- Diferentes heads pueden especializarse en diferentes características")
    print("- La atención permite relaciones globales entre parches distantes")
    
    # Simular mapas de atención (para demostración)
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.flatten()
    
    for i in range(min(num_heads, len(axes))):
        # Crear patrón de atención simulado
        attn_map = np.random.rand(14, 14)  # 14x14 parches para imagen 224x224 (16x16 patches)
        # Crear un patrón con un foco central para simular atención
        center = np.array([7, 7])
        for x in range(14):
            for y in range(14):
                dist = np.sqrt((x - 7)**2 + (y - 7)**2)
                attn_map[x, y] = np.exp(-dist / 3) + 0.1 * np.random.rand()
        axes[i].imshow(attn_map, cmap='hot', interpolation='nearest')
        axes[i].set_title(f'Attention Head {i+1}')
        axes[i].axis('off')
    
    plt.suptitle('Mapas de Atención - Vision Transformer', fontsize=14)
    plt.tight_layout()
    plt.show()

# Ejecutar visualización conceptual
visualize_attention_heads(model, None)

## Comparación: ViT vs CNN

| Característica | CNN (ResNet) | ViT |
|----------------|--------------|-----|
| Mecanismo principal | Convoluciones | Self-Attention |
| Campo receptivo | Local (crece con capas) | Global (desde el inicio) |
| Relaciones de largo alcance | Limitadas | Excelentes |
| Escalabilidad | Se degrada con profundidad | Mejora con más datos |
| Inductivo bias | Fuerte (localidad, traslación) | Débil (aprende todo) |
| Rendimiento con pocos datos | Mejor | Peor |
| Rendimiento con muchos datos | Limitado | Superior |
| Parámetros (base) | ~25M (ResNet50) | ~86M (ViT-Base) |

### ¿Cuándo usar ViT?
- **Grandes volúmenes de datos**: ImageNet, JFT-300M, conjuntos propietarios grandes
- **Tareas que requieren razonamiento global**: Detección de relaciones sutiles en la imagen
- **Transfer learning**: Pre-entrenar en ImageNet-21k y fine-tunear
- **Multimodalidad**: Misma arquitectura para texto e imagen

### ¿Cuándo usar CNN?
- **Pocos datos**: CNNs generalizan mejor con datasets pequeños
- **Recursos limitados**: Modelos más pequeños y eficientes
- **Tareas con fuerte inducción espacial**: Segmentación, detección con estructuras locales
- **Inferencia rápida**: CNNs como MobileNet son muy eficientes

In [ ]:
# ==================== GUARDADO DEL MODELO ====================

# Guardar el modelo fine-tuneado
def save_model(trainer, path="./vit_cifar10_final"):
    """Guarda el modelo y el feature extractor"""
    trainer.save_model(path)
    feature_extractor.save_pretrained(path)
    print(f"Modelo guardado en: {path}")

# Para guardar después del entrenamiento, descomentar:
# save_model(trainer)

# Cargar modelo guardado
def load_model(model_path="./vit_cifar10_final"):
    """Carga un modelo previamente guardado"""
    loaded_model = ViTForImageClassification.from_pretrained(model_path)
    loaded_feature_extractor = ViTFeatureExtractor.from_pretrained(model_path)
    return loaded_model, loaded_feature_extractor

print("Código para guardar/cargar modelos listo")

## Ejercicios Propuestos

### Ejercicio 1: Explorar diferentes tamaños de ViT
Prueba con diferentes versiones de ViT:
- `google/vit-base-patch16-224-in21k` (Base)
- `google/vit-large-patch16-224-in21k` (Large)
- `google/vit-small-patch32-224-in21k` (Small, parche 32x32)

Compara el rendimiento y el tiempo de entrenamiento.

### Ejercicio 2: Fine-tuning en otro dataset
Cambia el dataset a:
- `food101` (clasificación de alimentos)
- `beans` (clasificación de enfermedades de plantas)
- Tu propio dataset personalizado

### Ejercicio 3: Extraer embeddings
Usa el ViT para extraer características (sin la cabeza de clasificación) y usa esas características con otros clasificadores (SVM, Random Forest).

### Ejercicio 4: Visualización de atención real
Implementa hooks para extraer las matrices de atención reales del modelo durante la inferencia y visualiza qué regiones de la imagen atiende el modelo.

### Ejercicio 5: Comparación con CNN
Entrena un ResNet50 o EfficientNetB0 en CIFAR-10 y compara:
- Exactitud
- Tiempo de entrenamiento
- Número de parámetros
- Capacidad de generalización

## Recursos y Referencias

### Papers Fundamentales
1. **ViT Original**: Dosovitskiy et al., "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale" (2021)
2. **DeiT**: Touvron et al., "Training data-efficient image transformers & distillation through attention" (2021)
3. **Swin Transformer**: Liu et al., "Swin Transformer: Hierarchical Vision Transformer using Shifted Windows" (2021)

### Repositorios
- Hugging Face Transformers: https://github.com/huggingface/transformers
- ViT PyTorch (original): https://github.com/google-research/vision_transformer
- Awesome Vision Transformers: https://github.com/DA-southampton/Awesome-Vision-Transformer

### Artículos y Tutoriales
- [The Annotated Transformer](http://nlp.seas.harvard.edu/2018/04/03/attention.html)
- [Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/)
- [Vision Transformer Guide](https://huggingface.co/docs/transformers/model_doc/vit)

### Video Recomendados
- Yannic Kilcher's ViT paper review
- Two Minute Papers - Vision Transformer
- Umar Jamil's Transformer tutorial

## Conclusión

El Vision Transformer representa un cambio paradigmático en visión por computadora:

✅ **Ventajas capturadas**:
- Relaciones globales desde el primer momento
- Mejor escalabilidad con grandes datasets
- Arquitectura unificada con NLP
- State-of-the-art en múltiples tareas

⚠️ **Limitaciones a considerar**:
- Requiere muchos datos o fine-tuning cuidadoso
- Mayor costo computacional que CNNs equivalentes
- Menor eficiencia para imágenes muy grandes
- Inductive bias débil (debe aprender todo desde cero)

El futuro apunta a **modelos híbridos** (CNN + Transformer) como Swin Transformer, y a la integración con **LLMs multimodales** (GPT-4V, LLaVA, etc.).